# QUICKSTART

This section runs through the API for common tasks in machine learning.

## Working with data

PyTorch has two [primitives to work with data](https://pytorch.org/docs/stable/data.html): ``torch.utils.data.DataLoader`` and ``torch.utils.data.Dataset``.``Dataset`` stores the samples and their corresponding labels, and ``DataLoader`` wraps an iterable around the Dataset.

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
# from torchvision.transforms import ToTensor
from torchvision import transforms # for applying other transformations.

PyTorch offers domain-specific libraries such as [TorchText](https://pytorch.org/text/stable/index.html), [TorchVision](https://pytorch.org/vision/stable/index.html), and [TorchAudio](https://pytorch.org/audio/stable/index.html), all of which include datasets. In this follow up notebook, we will be using a TorchVision dataset.

The ``torchvision.datasets`` module contains ``Dataset`` objects for many real-world vision data like CIFAR, COCO ([full list here](https://pytorch.org/vision/stable/datasets.html)). In this follow-up notebook, we use the FashionMNIST dataset. Every TorchVision Dataset includes two arguments: ``transform`` and ``target_transform`` to modify the samples and labels respectively.

In [2]:
# Creating transformation that will be applied while loading the datasets
T = transforms.Compose([
    transforms.RandomRotation(0.25),
    transforms.ToTensor(),
])

In [3]:
# Downloading training datasets from open datasets
train_data = datasets.FashionMNIST(
    root="data", train=True,
    download=True, transform=T
)

In [4]:
# Downloading test datasets from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transforms.ToTensor() # not applying 'T' because it's test datasets! xD
)

We pass the ``Dataset`` as an argument to ``DataLoader``. This wraps an iterable over our dataset, and supports automatic batching, sampling, shuffling and multiprocess data loading. Here we define a batch size of 64, i.e. each element in the dataloader iterable will return a batch of 64 features and labels.

**Setting a batch size is important**
major reasons:
* when working with huge data it will get you cuda not allocate error! means running out of memory in short.
        
* gradient will get updated after completion of each batch training loop


In [5]:
batch_size = 64

# Creating data loaders

train_loader = DataLoader(train_data, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)

In [6]:
# checking shape
i = 0
for x, y in train_loader:
    print(f"{'-' * 23}[Batch no. {i + 1}]{'-' * 23}")
    print(f"Shape of X [Sample Numbers(N), Channel dims(C), H, W]: {x.shape}")
    print(f"Shape if y labels: {y.shape} and its dtype: {y.dtype}")
    print(f"{'-' * 30}{'-' * 30}", end="\n\n\n")

    if i > 5:
        break
    i += 1

-----------------------[Batch no. 1]-----------------------
Shape of X [Sample Numbers(N), Channel dims(C), H, W]: torch.Size([64, 1, 28, 28])
Shape if y labels: torch.Size([64]) and its dtype: torch.int64
------------------------------------------------------------


-----------------------[Batch no. 2]-----------------------
Shape of X [Sample Numbers(N), Channel dims(C), H, W]: torch.Size([64, 1, 28, 28])
Shape if y labels: torch.Size([64]) and its dtype: torch.int64
------------------------------------------------------------


-----------------------[Batch no. 3]-----------------------
Shape of X [Sample Numbers(N), Channel dims(C), H, W]: torch.Size([64, 1, 28, 28])
Shape if y labels: torch.Size([64]) and its dtype: torch.int64
------------------------------------------------------------


-----------------------[Batch no. 4]-----------------------
Shape of X [Sample Numbers(N), Channel dims(C), H, W]: torch.Size([64, 1, 28, 28])
Shape if y labels: torch.Size([64]) and its dtype:

Read more about [loading data in PyTorch](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html).

## Creating Models

To define a neural network in PyTorch, we create a class that inherits from [nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). We define the layers of the network in the ``__init__`` function and specify how data will pass through the network in the ``forward`` function. To accelerate operations in the neural network, we move it to the GPU if available.

In [7]:
# get gpu or cpu (it changes model platform for using either cpu of gpu)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [8]:
# Define model

class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.flatten = nn.Flatten() # for converting 2d img matrix to 1d vector
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512), # img size * image size
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [9]:
model = NeuralNet().to(device)
print(model)

NeuralNet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Read more about [building neural networks in PyTorch](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).

## Optimizing the Model Parameters

To train a model, we need a [loss function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an [optimizer](https://pytorch.org/docs/stable/optim.html).

In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training dataset (fed to it in batches), and backpropagates the prediction error to adjust the model’s parameters.

In [11]:
import numpy as np

def train(data_loader, model, loss_fn, optimizer):
    """
    for training deep learning models
    """
    size = len(data_loader.dataset)
    model.train()
    for batch, (x, y) in enumerate(data_loader):
        x, y = x.to(device), y.to(device)

        # Compute prediction error
        yh = model(x)
        loss = loss_fn(yh, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")                

We also check the model’s performance against the test dataset to ensure it is learning.

In [12]:
def test(dataloader, model, loss_fun):
    size = len(dataloader.dataset)
    batches = len(dataloader)
    model.eval()
    correct = 0
    test_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y  = x.to(device), y.to(device)
            yh = model(x)
            test_loss += loss_fn(yh, y).item()
            correct += (yh.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, Avg loss: {test_loss:>8f}\n")

The training process is conducted over several iterations (epochs). During each epoch, the model learns parameters to make better predictions. We print the model’s accuracy and loss at each epoch; we’d like to see the accuracy increase and the loss decrease with every epoch.

In [13]:
epochs = 15
for t in range(epochs):
    print(f"Epoch {t + 1}\n{'-' * 72}")
    train(train_loader, model, loss_fn, optimizer)
    test(test_loader, model, loss_fn)
print("Completed")

Epoch 1
------------------------------------------------------------------------
loss: 2.306743  [    0/60000]
loss: 2.296382  [ 6400/60000]
loss: 2.275495  [12800/60000]
loss: 2.270987  [19200/60000]
loss: 2.258036  [25600/60000]
loss: 2.222182  [32000/60000]
loss: 2.232318  [38400/60000]
loss: 2.201508  [44800/60000]
loss: 2.196068  [51200/60000]
loss: 2.157633  [57600/60000]
Test Error: 
 Accuracy: 47.7%, Avg loss: 2.159375

Epoch 2
------------------------------------------------------------------------
loss: 2.169499  [    0/60000]
loss: 2.159319  [ 6400/60000]
loss: 2.106526  [12800/60000]
loss: 2.121440  [19200/60000]
loss: 2.067073  [25600/60000]
loss: 2.016460  [32000/60000]
loss: 2.032949  [38400/60000]
loss: 1.965713  [44800/60000]
loss: 1.962490  [51200/60000]
loss: 1.882881  [57600/60000]
Test Error: 
 Accuracy: 60.3%, Avg loss: 1.889163

Epoch 3
------------------------------------------------------------------------
loss: 1.923680  [    0/60000]
loss: 1.891112  [ 6400/60

Read more about [Training your model](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).

## Saving Models

A common way to save a model is to serialize the internal state dictionary (containing the model parameters).

In [15]:
import os

try:
    torch.save(model.state_dict(), "sav_model/model_state_dict.pth")
except:
    os.makedirs("sav_model")
    torch.save(model.state_dict(), "sav_model/model_state_dict.pth")

## Loading Models

The process for loading a model includes re-creating the model structure and loading the state dictionary into it.

In [16]:
model = NeuralNet()
model.load_state_dict(torch.load("sav_model/model_state_dict.pth"))

<All keys matched successfully>

This model can now be used to make predictions.

In [19]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]


model.eval()
x, y = test_data[11][0], test_data[11][1] # using the 11th data point from the datasets
with torch.no_grad():
    yh = model(x)
    predicted, actual = classes[yh[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Sandal", Actual: "Sandal"


Read more about [Saving & Loading your model](https://pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html).

# Thank you for reading the notebook :-)